In [ ]:
%load_ext autoreload
%autoreload 2
import pandas as pd
import pickle
from core.preprocessing import UnscaledInputDictGenerator, InputVisualization, AggregatedInputDictGenerator, AggregatedInputVisualization
from core.helpers import get_experiment_id, get_config_file
import os

PROJECT_FILE = 'project.yaml'
CASE = 'dsm_Reinert_LP'
ROOT = os.path.dirname(os.path.dirname(os.getcwd()))
DATA_IN = os.path.join(ROOT, 'data', CASE, 'Input_Data')
EXPERIMENT_ID = get_experiment_id(DATA_IN, PROJECT_FILE)
CONFIG = get_config_file(DATA_IN, PROJECT_FILE)

DATA_COMPUTED_IN = os.path.join(ROOT, 'data', CASE, 'ComputedInput_Data', EXPERIMENT_ID)
DATA_OUT = os.path.join(ROOT, 'data', CASE, 'Output_Data', EXPERIMENT_ID, )
os.makedirs(DATA_COMPUTED_IN, exist_ok=True)
os.makedirs(DATA_OUT, exist_ok=True)

In [ ]:
# This cell is intended to be executed with 'load_tsa_inputs'=False in config.yaml.
# Thus, unscaled_input_dict should not contain aggregated data.

unscaled_generator = UnscaledInputDictGenerator(
    data_in_path = DATA_IN,
    project_file=PROJECT_FILE,
    objective= 'design'
)

unscaled_input_dict = unscaled_generator.generate_unscaled_input_dict()
print(f"len(time_slices)={len(unscaled_input_dict['time_slices'])}. (should be 8760)")

In [ ]:
# This cell is intended to be executed with 'load_tsa_inputs'=True in config.yaml.
# By setting ignore_tsa = True (see below), no tsa_inputs are loaded (so the config value is overwritten)
# Thus, unscaled_input_dict should not contain aggregated data.

unscaled_generator = UnscaledInputDictGenerator(
    data_in_path = DATA_IN,
    project_file=PROJECT_FILE,
    objective= 'design',
    ignore_tsa = True
)

unscaled_input_dict = unscaled_generator.generate_unscaled_input_dict()
print(f"len(time_slices)={len(unscaled_input_dict['time_slices'])}. (should be 8760)")

In [ ]:
# This cell shows the intended use of tsa. Please set 'load_tsa_inputs'=True in config.yaml.
# 1. generate an unscaled input dict while setting ignore_tsa=True
unscaled_input_dict_old = UnscaledInputDictGenerator(
    data_in_path = DATA_IN,
    project_file=PROJECT_FILE,
    objective= 'design',
    ignore_tsa = True
).generate_unscaled_input_dict()

# 2. aggregate the input dict and export the timeseries data
aggregator = AggregatedInputDictGenerator(
    data_in_path = DATA_IN,
    project_file=PROJECT_FILE,
    unaggregated_input_dict= unscaled_input_dict_old
)
agg_dict = aggregator.generate_aggregated_input_dict()
aggregator.export_tsa_to_yaml()

# 3. Now, the aggregated timeseries are loaded (if load_tsa_inputs in config.yaml is True)
unscaled_input_dict_new = UnscaledInputDictGenerator(
    data_in_path = DATA_IN,
    project_file=PROJECT_FILE,
    objective= 'design'
).generate_unscaled_input_dict()

In [ ]:
InputVisualization(
    DATA_IN, PROJECT_FILE
).visualize_unaggregated_input()

In [ ]:
AggregatedInputVisualization(
    agg_dict,
    visualization_mode='Line'
).visualize_aggregated_input()

In [ ]:
len(unscaled_input_dict['time_slices'])